In [74]:
import pandas as pd
import duckdb
import os
from datetime import datetime

In [75]:
source_path=r"E:\data_engineer\DEMovies\datalake"
dest_path=r"E:\data_engineer\DEMovies\bronze_layer"
db_path=r"E:\data_engineer\DEMovies\meta.duckdb"

In [76]:
conn= duckdb.connect(db_path)

In [77]:
conn.execute("""
CREATE TABLE IF NOT EXISTS file_metadata(
    filename VARCHAR PRIMARY KEY,
    last_modified TIMESTAMP,
    load_time TIMESTAMP,
    status VARCHAR
)
""")

In [78]:
for file in os.listdir(source_path):
    file_path= os.path.join(source_path,file)

    try:
        mod_date= datetime.fromtimestamp(
            os.path.getmtime(file_path)
        )

        result= conn.execute("""
            SELECT last_modified
            FROM file_metadata
            WHERE filename=?
            AND status='SUCCESS'
        """,[file]).fetchone()


        if result and mod_date<=result[0]:
            print("No new changes. skipping file")
            continue

        if file.lower().endswith(".csv"):
            df= pd.read_csv(file_path,encoding='latin1')
        else:
            df=pd.read_excel(file_path)

        row_count=len(df)
        load_time=datetime.utcnow()

        df['load_date']=load_time
        df['file_name']=file

        file_name=os.path.splitext(file)[0]
        extension=os.path.splitext(file)[1].lower()

        bronze_name=f"{file_name}_bronze{extension}"
        bronze_path=os.path.join(dest_path,bronze_name)

        if extension=='.csv':
            df.to_csv(bronze_path,index=False)
        else:
            df.to_excel(bronze_path,index=False)

        # save checkpoint after successful output
        conn.execute("""
            INSERT INTO file_metadata
            VALUES(?,?,?,'SUCCESS')
            ON CONFLICT(filename) DO UPDATE SET
                last_modified= EXCLUDED.last_modified,
                load_time=EXCLUDED.last_modified,
                status=EXCLUDED.status
        
        """,[file,mod_date,load_time])

        if result:
            print(f"Changed file reprocessed: {file}")
        else:
            print(f"New file added: {file}")

        print(f"Rows added: {row_count}")
        print(f"{bronze_name} created successfully")

    except Exception as e:
        print(f"Error processing {file}:{e}")

metada_df=conn.execute("""
    SELECT *
    FROM file_metadata
    ORDER BY filename
""").fetchdf()

display(metada_df)
conn.close()


No new changes. skipping file
Changed file reprocessed: imdb_movies_20261008.csv
Rows added: 3010
imdb_movies_20261008_bronze.csv created successfully
No new changes. skipping file


C:\Users\ExPertComputer\AppData\Local\Temp\ipykernel_6068\2568065476.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  load_time=datetime.utcnow()


,filename,last_modified,load_time,status
0,imdb_movies_20261007.csv,2026-10-10 17:41:15.875373,2026-10-10 17:41:15.875373,SUCCESS
1,imdb_movies_20261008.csv,2026-10-10 18:49:24.240139,2026-10-10 18:49:24.240139,SUCCESS
2,imdb_movies_20261009.csv,2026-10-10 14:29:45.958025,2026-10-10 11:43:03.383383,SUCCESS
